<a href="https://colab.research.google.com/github/vankayalasiri5/GEN-AI/blob/main/Week2_LLM_Client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


In [2]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time
import requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


In [4]:
!ollama pull llama3.2

In [5]:
!ollama list

NAME               ID              SIZE      MODIFIED           
llama3.2:latest    a80c4f17acd5    2.0 GB    About a minute ago    


In [6]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-16T08:53:47.668638285Z","response":"Hello.","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,13],"total_duration":108805970502,"load_duration":57731587182,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":51028430000,"eval_count":3,"eval_duration":40968000}

In [7]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

In [8]:
!ls -a

.  ..


In [9]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [10]:
!pip install -q -r requirements.txt

In [11]:
from google import genai
print("google-genai imported OK")

google-genai imported OK


In [12]:

%%writefile llm_client.py

from abc import ABC, abstractmethod
from google import genai
import requests


# Common interface
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini cloud provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)


Writing llm_client.py


In [13]:

%%writefile main.py

import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("Gemini API key not found in .env")

    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)


Writing main.py


In [14]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: anyio-4.16.0, platformdirs-4.14.3, langsmith-0.14.17, typeguard-4.6.0
collected 0 items                                                              

============================ no tests ran in 0.01s =============================
ERROR: file or directory not found: test_client.py



In [15]:
%run main.py

Choose provider (gemini/ollama): ollama
Enter your question: explain AI in two sentances

LLM Response:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI technology uses complex algorithms and machine learning techniques to enable computers to analyze data, recognize patterns, and make predictions or take actions, with the goal of improving efficiency, accuracy, and customer experience.


In [16]:
!ls -la

total 28
drwxr-xr-x 4 root root 4096 Nov 16 08:56 .
drwxr-xr-x 1 root root 4096 Nov 16 08:48 ..
-rw-r--r-- 1 root root 1284 Nov 16 08:55 llm_client.py
-rw-r--r-- 1 root root  704 Nov 16 08:55 main.py
drwxr-xr-x 2 root root 4096 Nov 16 08:56 __pycache__
drwxr-xr-x 3 root root 4096 Nov 16 08:56 .pytest_cache
-rw-r--r-- 1 root root   43 Nov 16 08:54 requirements.txt
